# XAI — LayerCAM, GradCAM++, Occlusion
Runs all 3 explainability methods on the single best model the group selected, for one sample image per class. All the logic lives in `src/xai/` (`common.py`, `cam_methods.py`, `occlusion.py`) — this notebook just sets the config and calls it.

## 0. Colab setup

In [ ]:
import os
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    REPO_URL = 'https://github.com/USTH-B3-Projects/DL2026-Group5-Project30.git'
    REPO_DIR = '/content/DL2026-Group5-Project30'

    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}

    !pip install -q datasets grad-cam captum gdown

print('Working directory:', os.getcwd())

## 1. Download checkpoint

In [ ]:
import gdown
from pathlib import Path

from src.training.config import CHECKPOINT_DIR

CHECKPOINT_NAME = 'resnet50_warmup_seed2026_best.pt'
CHECKPOINT_FILE_ID = '1pke61YPrQXScSIqSPTMKHnL7r8Tpr9r9'   # from the Drive share link

Path(CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)
checkpoint_dest = Path(CHECKPOINT_DIR) / CHECKPOINT_NAME
gdown.download(id=CHECKPOINT_FILE_ID, output=str(checkpoint_dest), quiet=False)
print('Saved to', checkpoint_dest)

## 2. Import

In [ ]:
import os
import sys
from pathlib import Path

import torch
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src').exists():
    raise RuntimeError('Folder src not found. Please open the notebook from this repo')

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from torch.utils.data import DataLoader
from torchvision import transforms
from datasets import load_dataset

from src.training.config import DEVICE, RESULTS_FIGURES_DIR, NUM_CLASSES
from src.datasets.glaucoma_dataset import GlaucomaDataset, _drop_duplicates, IDX_TO_CLASS
from src.xai.common import load_model, denormalize
from src.xai.cam_methods import run_layercam, run_gradcam_plusplus
from src.xai.occlusion import run_occlusion

print('Project root:', PROJECT_ROOT)
print('Device:', DEVICE)

## 3. Load the selected final model: ResNet-50 (seed 2026)

In [ ]:
MODEL_NAME = 'resnet50_warmup'

model, target_layers = load_model(CHECKPOINT_NAME)
print(f'Loaded {MODEL_NAME} ({CHECKPOINT_NAME})')
print('Target layer(s) for CAM methods:', target_layers)

## 4. Grab one test-set sample per class and a few misclassified ones

- `resnet50` was trained on a custom 320x320 pipeline (not the shared 224x224 `build_dataloaders()`), so this rebuilds the same 320x320 eval transform + dedup here, to match exactly what the model actually saw during training/evaluation.
- While scanning for one sample per class, this also predicts every image and keeps the first `NUM_MISCLASSIFIED` cases where the prediction was wrong - heatmaps on a *wrong* prediction are usually the most informative for the report (e.g. did the model look at the right region but still misjudge severity, or did it look somewhere irrelevant).

In [ ]:
EVAL_IMAGE_SIZE = 320
eval_transform = transforms.Compose([
    transforms.Resize((EVAL_IMAGE_SIZE, EVAL_IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406], 
        std=[0.229, 0.224, 0.225]),
])

raw_dataset = load_dataset('moondream/glaucoma-detection')
test_split = _drop_duplicates(raw_dataset['test'])
test_data = GlaucomaDataset(test_split, transform=eval_transform)
test_loader = DataLoader(test_data, batch_size=1, shuffle=False)

NUM_MISCLASSIFIED = 3        # how many wrongly-predicted examples to also explain

samples_by_class = {}
misclassified_samples = []

for images, labels in test_loader:
    true_label = labels.item()
    input_tensor = images.to(DEVICE)

    with torch.no_grad():
        pred_label = model(input_tensor).argmax(dim=1).item()

    if true_label not in samples_by_class:
        samples_by_class[true_label] = (true_label, pred_label, images[0])

    if pred_label != true_label and len(misclassified_samples) < NUM_MISCLASSIFIED:
        misclassified_samples.append((true_label, pred_label, images[0]))   

    if len(samples_by_class) == NUM_CLASSES and len(misclassified_samples) >= NUM_MISCLASSIFIED:
        break

print('Classes collected:', [IDX_TO_CLASS[k] for k in sorted(samples_by_class)])
print(f'Misclassified examples found: {len(misclassified_samples)}')
for true_label, pred_label, _ in misclassified_samples:
    print(f'  True: {IDX_TO_CLASS[true_label]} -> Pred: {IDX_TO_CLASS[pred_label]}')

## 5. Run LayerCAM + GradCAM++ + Occlusion on each sample

Takes a list of `(true_label, pred_label, img_tensor)` and runs LayerCAM + GradCAM++ + Occlusion against each image's *predicted* class (so the heatmap always explains what the model actually decided, right or wrong). Saves the grid PNG into `results/figures/xai/`.

In [ ]:
save_dir = Path(RESULTS_FIGURES_DIR) / 'xai'
save_dir.mkdir(parents=True, exist_ok=True)

method_names = ['Original', 'LayerCAM', 'GradCAM++', 'Occlusion']

def explain_samples(samples, title, filename):
    n = len(samples)
    fig, axes = plt.subplots(n, 4, figsize=(4 * 4, 4 * n))
    if n == 1:
        axes = axes.reshape(1, -1)

    for row, (true_label, pred_label, img_tensor) in enumerate(samples):
        input_tensor = img_tensor.unsqueeze(0).to(DEVICE)
        rgb_float = denormalize(img_tensor)

        layercam_vis = run_layercam(model, target_layers, input_tensor, pred_label, rgb_float)
        gradcampp_vis = run_gradcam_plusplus(model, target_layers, input_tensor, pred_label, rgb_float)
        occlusion_vis = run_occlusion(model, input_tensor, pred_label, rgb_float)

        panels = [rgb_float, layercam_vis, gradcampp_vis, occlusion_vis]
        for col, (panel, name) in enumerate(zip(panels, method_names)):
            ax = axes[row, col]
            ax.imshow(panel)
            ax.axis('off')
            if row == 0:
                ax.set_title(name)

        correctness = 'correct' if pred_label == true_label else 'WRONG'
        axes[row, 0].set_ylabel(
            f'True: {IDX_TO_CLASS[true_label]}\nPred: {IDX_TO_CLASS[pred_label]} ({correctness})',
            rotation=0, labelpad=70, va='center',
        )

    fig.suptitle(title, y=1.02)
    fig.tight_layout()
    out_path = save_dir / filename
    fig.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved to {out_path}')
    return out_path

## 6. One correct sample per class

In [ ]:
explain_samples(
    [samples_by_class[k] for k in sorted(samples_by_class)],
    title=f'{MODEL_NAME} (320x320) — LayerCAM vs GradCAM++ vs Occlusion (one per class)',
    filename=f'{MODEL_NAME}_xai_comparison.png',
)

## 7. Misclassified examples

In [ ]:
if misclassified_samples:
    explain_samples(
        misclassified_samples,
        title=f'{MODEL_NAME} (320x320) — Misclassified examples',
        filename=f'{MODEL_NAME}_xai_misclassified.png',
    )
else:
    print('No misclassified examples found while scanning -- try increasing NUM_MISCLASSIFIED '
          'or check more of the test set, since this stops as soon as both quotas are filled.')